# Diabetes Risk Prediction: Machine Learning Pipeline

This notebook covers the Data Analytics phase of the project, including handling class imbalance with SMOTE-ENN, training XGBoost/Random Forest models, and model interpretability using SHAP.

In [ ]:
# Import dependencies
import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.combine import SMOTEENN
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
import shap
import seaborn as sns

# Set visualization style
sns.set_theme(style="whitegrid")

### 1. Data Loading and Splitting

In [ ]:
# Load the cleaned dataset from the Hadoop MapReduce pipeline
df = pd.read_csv('../data/clean_dataset.csv')

# Separate Features (X) and Target (y)
X = df.drop('Diabetes_binary', axis=1)
y = df['Diabetes_binary']

# Split into 80% Training and 20% Testing
# IMPORTANT: We only apply SMOTE to the training set later to prevent data leakage!
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")
print("\nOriginal Class Distribution in Training Set:")
print(y_train.value_counts(normalize=True))

### 2. Handling Class Imbalance (SMOTE-ENN)

In [ ]:
# Initialize SMOTE-ENN
smote_enn = SMOTEENN(random_state=42)

# Resample the training data
X_train_resampled, y_train_resampled = smote_enn.fit_resample(X_train, y_train)

print("Resampled Class Distribution in Training Set:")
print(y_train_resampled.value_counts(normalize=True))

### 3. Model Training (XGBoost & Random Forest)

In [ ]:
# Train XGBoost
xgb_model = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_model.fit(X_train_resampled, y_train_resampled)

# Train Random Forest
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_model.fit(X_train_resampled, y_train_resampled)

### 4. Evaluation

In [ ]:
def evaluate_model(model, name, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    print(f"--- {name} Evaluation ---")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print("------------------------\n")

evaluate_model(xgb_model, "XGBoost", X_test, y_test)
evaluate_model(rf_model, "Random Forest", X_test, y_test)

### 5. Model Interpretability (SHAP)

In [ ]:
# Generate SHAP values for the XGBoost model (using a sample to speed up computation)
explainer = shap.TreeExplainer(xgb_model)
# Using a subset of test data for faster SHAP rendering
shap_values = explainer.shap_values(X_test.sample(2000, random_state=42))

# Plot the SHAP summary plot (This is a great chart for your journal!)
shap.summary_plot(shap_values, X_test.sample(2000, random_state=42))